<a href="https://colab.research.google.com/github/CPTR295/Sample-LLMs/blob/main/FineTuning_Generation_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from transformers import AutoTokenizer
from datasets import load_dataset

template_tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

In [2]:
def format_prompt(example):
  chat = example['messages']
  prompt = template_tokenizer.apply_chat_template(chat,tokenize=False)
  return {"text":prompt}

In [3]:
dataset = load_dataset("HuggingFaceH4/ultrachat_200k",  split="test_sft").shuffle(seed=42).select(range(3000))
dataset = dataset.map(format_prompt)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [4]:
print(dataset['text'][2])

<|user|>
Does the University of Pennsylvania offer any programs for non-traditional students?</s>
<|assistant|>
Yes, the University of Pennsylvania offers several programs for non-traditional students, including:

1. Penn LPS Online: This program offers online courses and degree programs for non-traditional students who wish to earn a degree from Penn. It offers several undergraduate and graduate degree programs.

2. College of Liberal and Professional Studies: This program offers a variety of degree programs, including full-time, part-time, online, and on-campus options. It is designed for working professionals and those who wish to complete their degree later in life.

3. Executive Education: This program offers short-term, intensive courses for working professionals who wish to enhance their skills and knowledge in their field.

4. Summer Sessions: This program offers summer courses for undergraduate and graduate students, including non-traditional students who wish to complete thei

In [5]:
import torch
from transformers import AutoModelForCausalLM,BitsAndBytesConfig

model_name= "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"


In [6]:
bnb_config = BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_compute_dtype=torch.bfloat16,bnb_4bit_use_double_quant=True)


In [7]:
!pip install -U bitsandbytes

In [8]:
!pip show bitsandbytes
!python --version

import sys
print(sys.executable)

import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

Name: bitsandbytes
Version: 0.50.2
Summary: k-bit optimizers and matrix multiplication routines.
Home-page: https://github.com/bitsandbytes-foundation/bitsandbytes
Author: 
Author-email: Tim Dettmers <dettmers@cs.washington.edu>
License: 
Location: /usr/local/lib/python3.13/dist-packages
Requires: numpy, packaging, torch
Required-by: 
Python 3.13.15
/usr/bin/python3
Torch: 2.11.0+cu128
CUDA available: True
CUDA version: 12.8


In [9]:
import bitsandbytes as bnb

print("bitsandbytes version:", bnb.__version__)
print("Successfully imported bitsandbytes!")

bitsandbytes version: 0.50.2
Successfully imported bitsandbytes!


In [10]:
from transformers.utils import is_bitsandbytes_available

print(is_bitsandbytes_available())

True


In [11]:
import transformers.utils.import_utils as iu
import inspect

print(inspect.getsource(iu.is_bitsandbytes_available))

@lru_cache
@_make_compile_constant
def is_bitsandbytes_available(min_version: str = BITSANDBYTES_MIN_VERSION) -> bool:
    is_available, bitsandbytes_version = _is_package_available("bitsandbytes", return_version=True)
    return is_available and version.parse(bitsandbytes_version) >= version.parse(min_version)



In [12]:
model = AutoModelForCausalLM.from_pretrained(model_name,device_map="auto",quantization_config=bnb_config)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [13]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [14]:
model.config.use_cache = False
model.config.pretraining_tp=1

In [15]:
tokenizer = AutoTokenizer.from_pretrained(model_name,trust_remote_code=False)
tokenizer.pad_token = "<PAD>"
tokenizer.padding_size = "left"

## Configuration

In [16]:
#LORA Configuration
from peft import LoraConfig,prepare_model_for_kbit_training,get_peft_model

In [17]:
peft_config = LoraConfig(
    lora_alpha=32, #Scaling
    lora_dropout=0.1,
    r=64,#Rank
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['k_proj', 'gate_proj', 'v_proj', 'up_proj', 'q_proj', 'o_proj', 'down_proj'] #Layers to target
)

In [18]:
import torchao
print(torchao.__version__)

0.18.0


In [19]:
!pip install -U torchao>0.16.0

In [20]:
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model,peft_config)

In [21]:
#training configuration
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir= "/",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    num_train_epochs=1,
    logging_steps=10,
    fp16=True,
    gradient_checkpointing=True,

)

In [22]:
!pip install -U trl

In [23]:
#Training
from trl import SFTTrainer

In [ ]:
from trl import SFTTrainer

# Set supervised fine-tuning parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    tokenizer=tokenizer,
    args=training_args,
    max_seq_length=512,

    # Leave this out for regular SFT
    peft_config=peft_config,
)

# Train model
trainer.train()


In [ ]:
trainer.train()

In [ ]:
trainer.model.save_pretrained("TinyLlama.1B-qlora")